# Day 2 – Text Representations: TF-IDF & Embeddings

## Sentiment Classification in Google Play Store User Reviews

**Dataset:** Google Play Store User Reviews  
**NLP Task:** Sentiment Classification  
**Text column:** `Translated_Review`  
**Target column:** `Sentiment`  
**Core model used previously:** DistilBERT Transformer

### Day 2 Objective

In Day 1, the review text was preprocessed and cleaned. In this notebook, we convert the preprocessed text to numerical form by comparing two major techniques:

1. **TF-IDF** – a sparse representation depending on word significance.
2. **Word Embeddings** – dense vectors encoding the semantic meaning of words.

We will link these techniques with the project's previous work on **LSTM/DistilBERT** models and indicate which one is best for sentiment classification.

## Learning Objectives

At the end of this notebook, we should be able to:

- Represent preprocessed text as numeric vectors with TF-IDF.
- Understand Bag-of-Words and TF-IDF weights.
- Implement a basic TF-IDF + Logistic Regression model baseline.
- Assess the model baseline with Accuracy, Precision, Recall, and F1-score.
- Describe word embeddings and semantic space.
- Import pre-trained embeddings (Word2Vec style) and find their closest neighbors.
- Compare TF-IDF and word embeddings.
- Describe contextual embeddings (Transformers).
- Compare TF-IDF with our previous LSTM and DistilBERT models in this project using the same metrics.
- Decide on the representation for our project.

#  Environment Setup

We use Google Colab with Python, Pandas, scikit-learn, NLTK, Gensim, and Transformers.

The same dataset source used in Day 1 is downloaded automatically with `kagglehub`.

In [1]:
!pip -q install kagglehub nltk scikit-learn gensim transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 72.3 MB/s eta 0:00:00


In [2]:
import os
import re
import string
import warnings
import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


True

#  Load the Google Play Reviews Dataset

This is the same dataset used in Day 1:

`googleplaystore_user_reviews.csv`

We use:

- `Translated_Review` → input text
- `Sentiment` → target label

In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("lava18/google-play-store-apps")

print("Dataset downloaded to:")
print(dataset_path)

print("\nFiles found:")
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

Using Colab cache for faster access to the 'google-play-store-apps' dataset.
Dataset downloaded to:
/kaggle/input/google-play-store-apps

Files found:
/kaggle/input/google-play-store-apps/googleplaystore.csv
/kaggle/input/google-play-store-apps/license.txt
/kaggle/input/google-play-store-apps/googleplaystore_user_reviews.csv


In [4]:
reviews_path = None

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower() == "googleplaystore_user_reviews.csv":
            reviews_path = os.path.join(root, file)
            break
    if reviews_path is not None:
        break

if reviews_path is None:
    raise FileNotFoundError("googleplaystore_user_reviews.csv was not found.")

reviews_df = pd.read_csv(reviews_path)

print("Dataset shape:", reviews_df.shape)
display(reviews_df.head())

Dataset shape: (64295, 5)


,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [5]:
df = reviews_df[["Translated_Review", "Sentiment"]].copy()

print("Missing values:")
display(df.isnull().sum())

print("\nSentiment distribution:")
display(df["Sentiment"].value_counts(dropna=False))

Missing values:


,0
Translated_Review,26868
Sentiment,26863



Sentiment distribution:


,count
Sentiment,
NaN,26863
Positive,23998
Negative,8271
Neutral,5163


In [6]:
df = df.dropna(subset=["Translated_Review", "Sentiment"]).copy()
df["Translated_Review"] = df["Translated_Review"].astype(str)

print("Shape after removing missing values:", df.shape)
display(df.head())

Shape after removing missing values: (37427, 2)


,Translated_Review,Sentiment
0,I like eat delicious food. That's I'm cooking ...,Positive
1,This help eating healthy exercise regular basis,Positive
3,Works great especially going grocery store,Positive
4,Best idea us,Positive
5,Best way,Positive


#  Recreate the Day 1 Cleaned Text

Day 1 created a traditional preprocessing pipeline that:

- lowercases text
- tokenizes words
- removes punctuation/non-alphabetic tokens
- removes stop words
- preserves `not`, `no`, and `never`
- lemmatizes words

We recreate that pipeline here so this notebook can run independently in Colab while staying consistent with Day 1.

In [7]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
negation_words = {"not", "no", "never"}
stop_words_for_sentiment = stop_words - negation_words

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    tokens = word_tokenize(text)

    cleaned_tokens = []

    for token in tokens:
        if not token.isalpha():
            continue

        if token in stop_words_for_sentiment:
            continue

        lemma = lemmatizer.lemmatize(token, pos="n")
        lemma = lemmatizer.lemmatize(lemma, pos="v")
        cleaned_tokens.append(lemma)

    return " ".join(cleaned_tokens)

df["clean_text"] = df["Translated_Review"].apply(preprocess_text)

df = df[df["clean_text"].str.strip().ne("")].copy()

print("Final usable rows:", len(df))
display(df[["Translated_Review", "clean_text", "Sentiment"]].head(10))

Final usable rows: 37353


,Translated_Review,clean_text,Sentiment
0,I like eat delicious food. That's I'm cooking ...,like eat delicious food cook food case best fo...,Positive
1,This help eating healthy exercise regular basis,help eat healthy exercise regular basis,Positive
3,Works great especially going grocery store,work great especially go grocery store,Positive
4,Best idea us,best idea u,Positive
5,Best way,best way,Positive
6,Amazing,amaze,Positive
8,"Looking forward app,",look forward app,Neutral
9,It helpful site ! It help foods get !,helpful site help food get,Neutral
10,good you.,good,Positive
11,Useful information The amount spelling errors ...,useful information amount spell error question...,Positive


## Quick Day 1 Verification

Before vectorization, we verify that the task-critical negations are still preserved.

In [8]:
negation_tests = [
    "I do not like this app.",
    "No, this application is not useful.",
    "I never recommend this app."
]

for sentence in negation_tests:
    cleaned = preprocess_text(sentence)
    print("Original:", sentence)
    print("Cleaned :", cleaned)
    print()

assert "not" in preprocess_text("I do not like this app.").split()
assert "no" in preprocess_text("No, I do not recommend this app.").split()
assert "never" in preprocess_text("I never recommend this app.").split()

print("Negation-preservation tests passed.")

Original: I do not like this app.
Cleaned : not like app

Original: No, this application is not useful.
Cleaned : no application not useful

Original: I never recommend this app.
Cleaned : never recommend app

Negation-preservation tests passed.


#  Train / Test Split

We split the **text before fitting TF-IDF**.

This is important because the vectorizer must learn its vocabulary and IDF statistics from the training set only. The test set is transformed using the already-fitted vectorizer.

This avoids data leakage.

In [9]:
from sklearn.model_selection import train_test_split

X_text = df["clean_text"]
y = df["Sentiment"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train_text))
print("Testing samples :", len(X_test_text))

print("\nTraining label distribution:")
display(y_train.value_counts())

print("\nTesting label distribution:")
display(y_test.value_counts())

Training samples: 29882
Testing samples : 7471

Training label distribution:


,count
Sentiment,
Positive,19194
Negative,6617
Neutral,4071



Testing label distribution:


,count
Sentiment,
Positive,4799
Negative,1654
Neutral,1018


# Bag-of-Words

Bag-of-Words represents a document using word counts while ignoring word order.

For example:

**Document 1:** `The movie was amazing`

**Document 2:** `The movie was boring`

The vocabulary may contain:

`The, movie, was, amazing, boring`

Each document is converted into a numeric vector based on how many times each word appears.

### Limitation

Bag-of-Words does not understand the meaning or relationship between words. It mainly records which words occur and how often.

For example, it treats **amazing** and **boring** as independent words without understanding that they express opposite sentiments.

TF-IDF improves this approach by assigning higher weights to words that are more distinctive across the corpus.

#  TF-IDF

**TF-IDF = Term Frequency × Inverse Document Frequency**

- **TF:** how frequently a word occurs in a document.
- **IDF:** gives less weight to words that occur in many documents.
- A word that is frequent in one document but rare across the corpus receives a higher TF-IDF weight.

This makes TF-IDF a strong, simple baseline for text classification.

### Important limitation

TF-IDF does **not** understand semantic similarity.

For example, `good`, `great`, and `excellent` are treated as separate features rather than automatically being recognized as semantically related.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000,min_df=2)
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)
print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape :", X_test_tfidf.shape)
print("Matrix type:", type(X_train_tfidf).__name__)

TF-IDF training shape: (29882, 5000)
TF-IDF testing shape : (7471, 5000)
Matrix type: csr_matrix


## Understanding the TF-IDF Matrix

The matrix is usually **sparse** because each review contains only a small portion of the full vocabulary.
For example, if there are 5,000 features, one review does not normally contain all 5,000 words.

In [11]:
feature_names = vectorizer.get_feature_names_out()
print("Number of TF-IDF features:", len(feature_names))
print("\nFirst 50 features:")
print(feature_names[:50])

Number of TF-IDF features: 5000

First 50 features:
['aap' 'ab' 'abandon' 'abc' 'abd' 'ability' 'abit' 'able' 'abroad'
 'abrupt' 'abruptly' 'absence' 'absolute' 'absolutely' 'absurd' 'abt'
 'abuse' 'abusive' 'acc' 'accent' 'accept' 'acceptable' 'access'
 'accessibility' 'accessible' 'accessory' 'accident' 'accidentally'
 'accidently' 'accommodation' 'accomplish' 'accord' 'account'
 'accountable' 'acct' 'accumulate' 'accuracy' 'accurate' 'accurately'
 'accuweather' 'ace' 'ache' 'achieve' 'achievement' 'acknowledge' 'acorn'
 'acoutered' 'acquire' 'acrobat' 'across']


In [18]:
sample_size = min(5, X_train_tfidf.shape[0])
sample_matrix = X_train_tfidf[:sample_size].toarray()
feature_scores = sample_matrix.sum(axis=0)
top_feature_indices = feature_scores.argsort()[-20:][::-1]
tfidf_sample = pd.DataFrame(sample_matrix[:, top_feature_indices],columns=feature_names[top_feature_indices])
display(tfidf_sample)

,rubbish,fake,effort,smart,alarm,back,save,ad,well,cover,star,nice,pop,work,close,second,wait,time,screen,fantastic
0,0.0,0.0,0.66817,0.000000,0.00000,0.000000,0.462864,0.000000,0.40248,0.000000,0.000000,0.000000,0.000000,0.306382,0.000000,0.000000,0.000000,0.288871,0.000000,0.0
1,1.0,0.0,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
2,0.0,1.0,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3,0.0,0.0,0.00000,0.000000,0.00000,0.465752,0.000000,0.449255,0.00000,0.392105,0.000000,0.000000,0.310162,0.000000,0.302275,0.294725,0.293255,0.000000,0.258973,0.0
4,0.0,0.0,0.00000,0.618077,0.58947,0.000000,0.000000,0.000000,0.00000,0.000000,0.368718,0.366815,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [19]:
def show_top_tfidf_terms(row_index, top_n=10):
    row = X_train_tfidf[row_index]
    scores = row.toarray().ravel()
    top_indices = scores.argsort()[::-1][:top_n]

    result = pd.DataFrame({
        "Term": feature_names[top_indices],
        "TF-IDF": scores[top_indices]
    })

    return result[result["TF-IDF"] > 0].reset_index(drop=True)

for i in range(min(3, X_train_tfidf.shape[0])):
    print(f"Review {i} top TF-IDF terms:")
    display(show_top_tfidf_terms(i, top_n=10))

Review 0 top TF-IDF terms:


,Term,TF-IDF
0,effort,0.668170
1,save,0.462864
2,well,0.402480
3,work,0.306382
4,time,0.288871


Review 1 top TF-IDF terms:


,Term,TF-IDF
0,rubbish,1.0


Review 2 top TF-IDF terms:


,Term,TF-IDF
0,fake,1.0


#  TF-IDF Baseline Classifier

Now we use the TF-IDF vectors as input to a simple **Logistic Regression** classifier.
Pipeline:
Cleaned Text → TF-IDF → Logistic Regression → Sentiment Prediction
This is our traditional NLP baseline.

In [20]:
from sklearn.linear_model import LogisticRegression
tfidf_model = LogisticRegression(max_iter=1000,random_state=42)
tfidf_model.fit(X_train_tfidf, y_train)
y_pred_tfidf = tfidf_model.predict(X_test_tfidf)
print("TF-IDF baseline training completed.")

TF-IDF baseline training completed.


In [21]:
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,classification_report,confusion_matrix)
tfidf_accuracy = accuracy_score(y_test, y_pred_tfidf)
tfidf_precision = precision_score(y_test, y_pred_tfidf, average="weighted", zero_division=0)
tfidf_recall = recall_score(y_test, y_pred_tfidf, average="weighted", zero_division=0)
tfidf_f1 = f1_score(y_test, y_pred_tfidf, average="weighted", zero_division=0)
print("TF-IDF + Logistic Regression Results")
print("------------------------------------")
print(f"Accuracy : {tfidf_accuracy}")
print(f"Precision: {tfidf_precision}")
print(f"Recall   : {tfidf_recall}")
print(f"F1-score : {tfidf_f1}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tfidf, zero_division=0))

TF-IDF + Logistic Regression Results
------------------------------------
Accuracy : 0.8980056217373845
Precision: 0.8969266816476079
Recall   : 0.8980056217373845
F1-score : 0.8967863008478514

Classification Report:
              precision    recall  f1-score   support

    Negative       0.88      0.79      0.83      1654
     Neutral       0.82      0.81      0.82      1018
    Positive       0.92      0.95      0.94      4799

    accuracy                           0.90      7471
   macro avg       0.87      0.85      0.86      7471
weighted avg       0.90      0.90      0.90      7471



In [24]:
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred_tfidf, labels=["Negative", "Neutral", "Positive"])
cm_df = pd.DataFrame(cm,index=["Actual Negative", "Actual Neutral", "Actual Positive"],columns=["Pred Negative", "Pred Neutral", "Pred Positive"])
print(cm_df)

Confusion Matrix:
                 Pred Negative  Pred Neutral  Pred Positive
Actual Negative           1312            86            256
Actual Neutral              46           825            147
Actual Positive            136            91           4572


#  What Insights Can We Gain From the TF-IDF Baseline?

The baseline provides us with an objective benchmark.
If a sophisticated model cannot beat such a basic benchmark, then there is a good chance that it is too complicated to justify for this application.
TF-IDF is attractive because it is:
- simple
- fast
- cheap to compute
- understandable
- powerful when used in text classification

The downside of the approach is that it doesn't model semantics or context.

#  Word Embeddings

Word embeddings use a different idea,instead of representing a word only by its frequency, a word is represented as a **dense vector**.
Conceptually:

good → [0.21, -0.13, 0.74, ...]

Words used in similar contexts tend to have vectors that are close together.
Classic pre-trained embedding approaches include:

- **Word2Vec**
- **GloVe**

This is the idea of **semantic geometry**: relationships between words can be represented by distances and directions in vector space.

## TF-IDF vs. Word Embeddings

| Property | TF-IDF | Word Embeddings |
|---|---|---|
| Representation | Sparse | Dense |
| Main information | Word importance | Word relationships / meaning |
| Semantic similarity | Limited | Stronger |
| Word order | Ignored | Depends on embedding/model |
| Speed | Usually fast | Usually more computationally involved |
| Good use | Strong baseline | Semantic / deep-learning input |

#  Load Pre-trained Word Embeddings

We use a pre-trained GloVe 100-dimensional model through Gensim.


In [26]:
import requests
import zipfile
import os

print("Downloading GloVe embeddings...")

url = "https://nlp.stanford.edu/data/glove.6B.zip"
response = requests.get(url, timeout=120)

with open("/content/glove.6B.zip", "wb") as f:
    f.write(response.content)

print("Download completed!")

Download completed!


In [27]:
with zipfile.ZipFile("/content/glove.6B.zip", "r") as zip_ref:
    zip_ref.extractall("/content/glove")

print("GloVe embeddings extracted successfully!")

GloVe embeddings extracted successfully!


In [28]:
import numpy as np

embedding_dim = 100
glove_path = f"/content/glove/glove.6B.{embedding_dim}d.txt"

glove_embeddings = {}

with open(glove_path, encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        glove_embeddings[word] = vector

print(f"Loaded {len(glove_embeddings):,} word vectors.")

Loaded 400,000 word vectors.


In [32]:
from gensim.models import KeyedVectors
embedding_model = KeyedVectors(vector_size=embedding_dim)
embedding_model.add_vectors(list(glove_embeddings.keys()),list(glove_embeddings.values()))

print("GloVe model is ready!")

GloVe model is ready!


#  Nearest Neighbors and Semantic Geometry

`most_similar()` finds words whose vectors are close to the selected word.

This lets us inspect whether semantically related words are located near one another in the learned vector space.

In [33]:
words_to_check = ["good", "bad", "happy", "great"]

for word in words_to_check:
    if word in embedding_model:
        print(f"Nearest neighbors for '{word}':")

        display(pd.DataFrame(
            embedding_model.most_similar(word, topn=5),
            columns=["Word", "Similarity"]
        ))
    else:
        print(f"'{word}' is not in the GloVe vocabulary.")

Nearest neighbors for 'good':


,Word,Similarity
0,better,0.893191
1,sure,0.831456
2,really,0.829776
3,kind,0.828827
4,very,0.826080


Nearest neighbors for 'bad':


,Word,Similarity
0,worse,0.792971
1,good,0.770280
2,things,0.765360
3,too,0.763015
4,thing,0.760967


Nearest neighbors for 'happy':


,Word,Similarity
0,'m,0.841329
1,feel,0.813258
2,'re,0.804808
3,i,0.793828
4,'ll,0.791627


Nearest neighbors for 'great':


,Word,Similarity
0,greatest,0.788265
1,good,0.759280
2,little,0.758575
3,much,0.747705
4,well,0.740101


## Word Analogy Demonstration

A classic example used to explain embedding geometry is:

king - man + woman ≈ queen

The exact result depends on the specific pre-trained model, but the experiment demonstrates how vector directions can encode relationships.

In [39]:
if all(word in embedding_model for word in ["king", "man", "woman"]):
    analogy = embedding_model.most_similar(
        positive=["king", "woman"],
        negative=["man"],
        topn=5
    )

    display(pd.DataFrame(analogy, columns=["Word", "Similarity"]))
else:
    print("One or more analogy words are not available in the embedding vocabulary.")

,Word,Similarity
0,queen,0.769854
1,monarch,0.684338
2,throne,0.675574
3,daughter,0.659456
4,princess,0.652053


#  Important Difference: Word Embeddings Are Not the Same as TF-IDF

TF-IDF creates a vector for a **document** based on the importance of its words.

Word embeddings normally create a vector for a **word**.

For a whole review, embeddings can be combined or passed into a neural architecture such as an LSTM.

Therefore, these are different representation strategies:

TF-IDF → document-level sparse features → classical ML

Word embeddings → word-level dense vectors → sequence/deep-learning models

# Contextual Embeddings

The meaning of a word can change depending on the context in which it appears.

Example:

- `The child is playing with a bat.`
- `The farmer saw a bat flying at night.`

The word **bat** refers to different things in these two sentences.

Transformer models such as **BERT / DistilBERT** generate contextual representations, allowing the representation of a word to depend on the surrounding words.

This makes contextual embeddings useful for understanding words with multiple meanings.

## DistilBERT Connection to the Project

The project's previous core model was **DistilBERT**.

The conceptual Transformer pipeline is:

`Review Text → DistilBERT Tokenizer → Token IDs + Attention Mask → DistilBERT → Classification Head → Sentiment`

Unlike the traditional TF-IDF pipeline, we should not aggressively remove stop words or manually alter the text before a Transformer unless there is a specific reason. The Transformer has its own tokenizer and relies on contextual information.

In [35]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
example_texts = ["This app is amazing!","I do not like this application."]
encoded = tokenizer(example_texts,padding=True,truncation=True,return_tensors="pt")

print("Tokenizer output keys:")
print(encoded.keys())

print("\nInput IDs shape:")
print(encoded["input_ids"].shape)

print("\nTokens for the first review:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer output keys:
KeysView({'input_ids': tensor([[  101,  2023, 10439,  2003,  6429,   999,   102,     0,     0],
        [  101,  1045,  2079,  2025,  2066,  2023,  4646,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])})

Input IDs shape:
torch.Size([2, 9])

Tokens for the first review:
['[CLS]', 'this', 'app', 'is', 'amazing', '!', '[SEP]', '[PAD]', '[PAD]']


#  Comparing TF-IDF with the Previous LSTM / DistilBERT Models

The Day 2 lab asks us to compare the TF-IDF model against the project's previous LSTM/Transformer using the **same evaluation metrics**.

The TF-IDF metrics below are calculated directly in this notebook.

The LSTM and DistilBERT values are entered from the project's previous experiment so that we do not need to retrain those larger models every time Day 2 is executed.

In [36]:

previous_results = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "Text LSTM",
        "DistilBERT"
    ],
    "Accuracy": [
        tfidf_accuracy,
        0.6340,
        0.8960
    ],
    "Weighted Precision": [
        tfidf_precision,
        0.4020,
        0.8953
    ],
    "Weighted Recall": [
        tfidf_recall,
        0.6340,
        0.8960
    ],
    "Weighted F1": [
        tfidf_f1,
        0.4920,
        0.8940
    ]
})

display(previous_results.round(4))

,Model,Accuracy,Weighted Precision,Weighted Recall,Weighted F1
0,TF-IDF + Logistic Regression,0.898,0.8969,0.898,0.8968
1,Text LSTM,0.634,0.4020,0.634,0.4920
2,DistilBERT,0.896,0.8953,0.896,0.8940


## Interpreting the Comparison

Use **Weighted F1** together with the other metrics rather than accuracy alone.

The comparison helps answer two questions:

1. How much can a simple TF-IDF baseline achieve?
2. Is the additional complexity of LSTM / DistilBERT justified?

The final choice should consider both **performance and computational cost**.

In [38]:
best_model = previous_results.loc[previous_results["Weighted F1"].idxmax()]
print("Best model by Weighted F1:")
print(best_model["Model"])
print(f"Weighted F1: {best_model['Weighted F1']:.4f}")

Best model by Weighted F1:
TF-IDF + Logistic Regression
Weighted F1: 0.8968


## TF-IDF vs. Embeddings

| Method                     | Advantages                                                        | Limitations                                        |
| -------------------------- | ----------------------------------------------------------------- | -------------------------------------------------- |
| **TF-IDF**                 | Fast, simple, interpretable, and strong baseline                  | Ignores meaning and word order                     |
| **Word Embeddings**        | Capture semantic relationships and work well with neural networks | Less contextual and more computationally demanding |
| **Transformer Embeddings** | Understand context and ambiguity effectively                      | More complex and resource-intensive                |

# Hands-On Lab: Vectorizing Text — Completed Tasks

- [x] **Step 1:** Applied TF-IDF to the cleaned Day 1 reviews and trained a Logistic Regression classifier as a baseline.

- [x] **Step 2:** Loaded pre-trained GloVe word embeddings and found the nearest neighbors of selected words to demonstrate semantic geometry.

- [x] **Step 3:** Compared the TF-IDF baseline with the previous LSTM and DistilBERT models using Accuracy, Precision, Recall, and Weighted F1-score.

- [x] **Step 4:** Selected DistilBERT as the core representation because it captures context effectively, while retaining TF-IDF as a fast and interpretable baseline.

## Final Representation Decision
TF-IDF is used as a fast baseline, while DistilBERT is selected as the core model because it captures context and achieved better results than LSTM.

TF-IDF: Baseline

Word Embeddings: Semantic representation

DistilBERT: Final core model

#  Day 2 Requirements Checklist

- [x] Explain how text becomes numeric vectors.
- [x] Explain Bag-of-Words.
- [x] Apply TF-IDF.
- [x] Inspect the TF-IDF vocabulary.
- [x] Inspect TF-IDF vectors.
- [x] Train a simple TF-IDF classifier.
- [x] Evaluate Accuracy, Precision, Recall, and F1-score.
- [x] Explain word embeddings.
- [x] Load pre-trained embeddings.
- [x] Find nearest neighbors for selected words.
- [x] Demonstrate semantic geometry with an analogy.
- [x] Compare TF-IDF and embeddings.
- [x] Explain contextual embeddings.
- [x] Connect contextual embeddings to DistilBERT.
- [x] Compare TF-IDF against previous LSTM / DistilBERT results.
- [x] Document the representation choice.

# Final Summary
Day 2 compared TF-IDF, word embeddings, and contextual embeddings for text representation.
TF-IDF is a fast and interpretable baseline, while DistilBERT remains the preferred model because it captures context more effectively.